# Real-estate Scraper (Voursa + Wassit)

This notebook scrapes real-estate listings from:

- **Voursa**: `https://voursa.com/EN/categories/real_estate` (supports a **View More** button)
- **Wassit**: `https://www.wassit.info/immobilier.html`

It outputs a cleaned **pandas DataFrame** and exports it to CSV/JSON.

> ⚠️ Scraping note: Respect each website’s Terms of Service and robots.txt, keep a reasonable rate-limit, and don’t run this in a tight loop.


In [2]:
# Core imports
import re
import time
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Optional (only needed for the 'click View More' approach)
USE_SELENIUM = True


In [3]:
# Networking helpers: requests Session with retries + a browser-like User-Agent
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9,fr;q=0.8,ar;q=0.7",
}

def make_session(
    total_retries: int = 5,
    backoff_factor: float = 0.6,
    status_forcelist: Tuple[int, ...] = (429, 500, 502, 503, 504),
) -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=total_retries,
        connect=total_retries,
        read=total_retries,
        status=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def fetch_html(session: requests.Session, url: str, timeout: int = 20) -> str:
    r = session.get(url, headers=DEFAULT_HEADERS, timeout=timeout)
    r.raise_for_status()
    return r.text


## Voursa scraper

Voursa’s category page contains listing links, and also exposes a **View More** button (which may load additional listings).

We provide two approaches:

1. **Requests-only**: quick, gets the server-rendered listings.
2. **Selenium**: clicks **View More** repeatedly to load more, then parses `driver.page_source`.


In [4]:
# Voursa parsing

VOURSA_LISTING_PATH_FRAGMENT = "/EN/categories/real_estate/ads/"
VOURSA_PROPERTY_TYPES = [
    "Residential Property",
    "Land Plot",
    "Store",
    "Office",
    "Warehouse",
    "Commercial and Industrial Real Estate",
    "Other",
]

KNOWN_LOCATIONS = [
    "Tevragh Zeina",
    "Dar Naim",
    "Arafat",
    "Teyarett",
    "Ksar",
    "Riyadh",
    "EL Mina",
    "Toujounine",
    "Sebkha",
    "Nouadhibou",
    "Chami"
]

_price_re = re.compile(r"(?P<price>\d[\d,]*)\s*MRU", re.IGNORECASE)
_time_re = re.compile(r"(?P<n>\d+)\s+(?P<u>hour|hours|day|days|month|months|year|years)\s+ago", re.IGNORECASE)

def parse_voursa_listing_text(text: str) -> Dict[str, Any]:
    """Heuristic parser for the listing anchor text."""
    raw = " ".join(text.split())
    out: Dict[str, Any] = {"raw_text": raw}

    # price
    m = _price_re.search(raw)
    if m:
        out["price_mru"] = int(m.group("price").replace(",", ""))
        prefix = raw[: m.start()].strip()
        suffix = raw[m.end() :].strip()
    else:
        out["price_mru"] = None
        prefix, suffix = raw, ""

    # property type
    prop_type = None
    for t in VOURSA_PROPERTY_TYPES:
        if t in suffix:
            prop_type = t
            break
    out["property_type"] = prop_type

    # time ago
    tm = _time_re.search(raw)
    out["time_ago"] = tm.group(0) if tm else None

    # split on bullet (location)
    if "•" in raw:
        left, right = [p.strip() for p in raw.split("•", 1)]
    else:
        left, right = raw, ""

    out["title_guess"] = prefix if m else left

    # Try extracting location from right
    tokens = right.split()

    # location heuristic
    loc = None
    for loc_candidate in KNOWN_LOCATIONS:
        lc_tokens = loc_candidate.split()
        if tokens[: len(lc_tokens)] == lc_tokens:
            loc = loc_candidate
            break
    out["location"] = loc

    return out

def parse_voursa_category_html(html: str, base_url: str = "https://voursa.com") -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    rows: List[Dict[str, Any]] = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if VOURSA_LISTING_PATH_FRAGMENT in href:
            text = a.get_text(" ", strip=True)
            parsed = parse_voursa_listing_text(text)
            parsed["source"] = "voursa"
            parsed["url"] = href if href.startswith("http") else base_url.rstrip("/") + href
            rows.append(parsed)

    df = pd.DataFrame(rows).drop_duplicates(subset=["url"])
    return df

In [5]:
# Override: ensure Voursa category parser returns a DataFrame

def parse_voursa_category_html(html: str, base_url: str = "https://voursa.com") -> pd.DataFrame:
    """Parse the Voursa real-estate category page into a DataFrame.

    This overrides the earlier definition to make sure we always return a
    de-duplicated DataFrame of listing rows.
    """
    soup = BeautifulSoup(html, "html.parser")
    rows: List[Dict[str, Any]] = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if VOURSA_LISTING_PATH_FRAGMENT in href:
            text = a.get_text(" ", strip=True)
            parsed = parse_voursa_listing_text(text)
            parsed["source"] = "voursa"
            parsed["url"] = href if href.startswith("http") else base_url.rstrip("/") + href
            rows.append(parsed)

    df = pd.DataFrame(rows).drop_duplicates(subset=["url"])
    return df


In [6]:
# Voursa scraping approach 1: requests-only

session = make_session()
voursa_url = "https://voursa.com/EN/categories/real_estate"

voursa_html = fetch_html(session, voursa_url)
voursa_df = parse_voursa_category_html(voursa_html)

print("Voursa listings (requests-only):", len(voursa_df))
voursa_df.head(10)


Voursa listings (requests-only): 51


,raw_text,price_mru,property_type,time_ago,title_guess,location,source,url
0,Grand plaza Tevragh Zeina • 9 months ago Share...,2500,None,9 months ago,Grand plaza Tevragh Zeina • 9 months ago Share...,None,voursa,https://voursa.com/EN/categories/real_estate/a...
2,"نيمرو للبيــع فالمطار سوق السيارات 2,000,000 M...",2000000,Land Plot,4 hours ago,نيمرو للبيــع فالمطار سوق السيارات,Dar Naim,voursa,https://voursa.com/EN/categories/real_estate/a...
3,"سكن فبوحديده احذ كدروه حظ سعيد 550,000 MRU Res...",550000,Residential Property,4 hours ago,سكن فبوحديده احذ كدروه حظ سعيد,Toujounine,voursa,https://voursa.com/EN/categories/real_estate/a...
4,"فلبوادي على شارع كبير 300م 1,400,000 MRU Land ...",1400000,Land Plot,4 hours ago,فلبوادي على شارع كبير 300م,Tevragh Zeina,voursa,https://voursa.com/EN/categories/real_estate/a...
5,"ديبلكس اكبير للبيع 15,000,000 MRU Residential ...",15000000,Residential Property,6 hours ago,ديبلكس اكبير للبيع,Tevragh Zeina,voursa,https://voursa.com/EN/categories/real_estate/a...
6,"دار فعرفات الداية 11 1,300,000 MRU Residential...",1300000,Residential Property,8 hours ago,دار فعرفات الداية 11,Arafat,voursa,https://voursa.com/EN/categories/real_estate/a...
7,"ديبلكس للبيع 3,300,000 MRU Residential Propert...",3300000,Residential Property,8 hours ago,ديبلكس للبيع,Teyarett,voursa,https://voursa.com/EN/categories/real_estate/a...
8,"نيمرو للبيــع في سكتير 5 F Nord 6,700,000 MRU ...",6700000,Land Plot,9 hours ago,نيمرو للبيــع في سكتير 5 F Nord,Tevragh Zeina,voursa,https://voursa.com/EN/categories/real_estate/a...
9,"150م² فكرن فشرم الشيخ البوادي نظيفة 1,800,000 ...",1800000,Land Plot,9 hours ago,150م² فكرن فشرم الشيخ البوادي نظيفة,Teyarett,voursa,https://voursa.com/EN/categories/real_estate/a...
10,"ديبلكس للبيع 8,500,000 MRU Residential Propert...",8500000,Residential Property,9 hours ago,ديبلكس للبيع,Teyarett,voursa,https://voursa.com/EN/categories/real_estate/a...


In [7]:
# Voursa scraping approach 2: Selenium for "View More" (optional)

def scrape_voursa_with_selenium(url: str, max_clicks: int = 50, wait_s: int = 10, click_sleep_s: float = 0.8) -> str:
    """Open the category page, click 'View More' until exhausted or max_clicks is reached, return full HTML."""
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException, StaleElementReferenceException, ElementClickInterceptedException

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1365,768")

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)

    clicks = 0
    while clicks < max_clicks:
        try:
            btn = WebDriverWait(driver, wait_s).until(
                EC.element_to_be_clickable((By.XPATH, "//*[normalize-space()='View More']"))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(0.2)
            btn.click()
            clicks += 1
            time.sleep(click_sleep_s)
        except (TimeoutException, StaleElementReferenceException, ElementClickInterceptedException):
            break

    html = driver.page_source
    driver.quit()
    print(f"Clicked View More {clicks} time(s)")
    return html

if USE_SELENIUM:
    voursa_html_full = scrape_voursa_with_selenium(voursa_url, max_clicks=80)
    voursa_df_full = parse_voursa_category_html(voursa_html_full)
    print("Voursa listings (selenium):", len(voursa_df_full))
else:
    voursa_df_full = voursa_df


Clicked View More 80 time(s)
Voursa listings (selenium): 4050


## Enrich listings with Overview & Details from detail pages

Each Voursa listing has a detail page with **Overview** (Lot Size, Street Size, Closest Landmark) and **Details** (Land Title, Property Type, Halls, Balconies, Bathrooms, Rooms). We fetch each URL and extract these into new columns.

In [8]:
# Parse Voursa detail page: extract Overview & Details from embedded JSON

import json

def parse_voursa_detail_html(html: str) -> Dict[str, Any]:
    """Extract overview and details from a Voursa listing detail page.

    The page embeds ad data as JSON in a script tag. Returns a flat dict with
    lot_size, street_size, closest_landmark, land_title, detail_property_type,
    halls, balconies, bathrooms, rooms.
    """
    out: Dict[str, Any] = {
        "lot_size": None,
        "street_size": None,
        "closest_landmark": None,
        "land_title": None,
        "detail_property_type": None,
        "halls": None,
        "balconies": None,
        "bathrooms": None,
        "rooms": None,
    }

    # Find ad JSON: contains "overview" and "details" (may use \" in HTML)
    idx = html.find('"overview"')
    if idx < 0:
        idx = html.find('\\"overview\\"')
    if idx < 0:
        return out

    # Find opening brace of the ad object (search backwards from overview)
    start = html.rfind("{", 0, idx + 1)
    if start < 0:
        return out

    # Match balanced braces
    depth = 0
    end = start
    for i in range(start, len(html)):
        c = html[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                end = i
                break

    try:
        raw = html[start : end + 1]
        raw = raw.replace('\\"', '"')  # Unescape JSON embedded in HTML
        data = json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        return out

    # Map overview key -> column
    for item in data.get("overview") or []:
        key = (item.get("key") or "").strip()
        val = (item.get("value") or "").strip()
        if key == "Lot Size":
            out["lot_size"] = val if val and val.upper() != "N/A" else None
        elif key == "Street Size":
            out["street_size"] = val if val and val.upper() != "N/A" else None
        elif key == "Closest Landmark":
            out["closest_landmark"] = val or None

    # Map details key -> column
    for item in data.get("details") or []:
        key = (item.get("key") or "").strip()
        val = (item.get("value") or "").strip()
        if key == "Land Title":
            out["land_title"] = val or None
        elif key == "Property Type":
            out["detail_property_type"] = val or None
        elif key == "Halls":
            out["halls"] = int(val) if val and val.isdigit() else None
        elif key == "Balconies":
            out["balconies"] = int(val) if val and val.isdigit() else None
        elif key == "Bathrooms":
            out["bathrooms"] = int(val) if val and val.isdigit() else None
        elif key == "Rooms":
            out["rooms"] = int(val) if val and val.isdigit() else None

    return out


def enrich_df_with_details(
    df: pd.DataFrame,
    session: requests.Session,
    url_col: str = "url",
    source_filter: str = "voursa",
    delay_s: float = 1.0,
    max_rows: Optional[int] = None,
) -> pd.DataFrame:
    """Fetch each listing's detail page and add Overview/Details columns."""
    if url_col not in df.columns:
        return df

    subset = df[df["source"] == source_filter] if "source" in df.columns else df
    urls = subset[url_col].drop_duplicates().tolist()
    if max_rows is not None:
        urls = urls[:max_rows]

    url_to_details: Dict[str, Dict[str, Any]] = {}
    for i, url in enumerate(urls):
        try:
            html = fetch_html(session, url, timeout=15)
            url_to_details[url] = parse_voursa_detail_html(html)
        except Exception as e:
            url_to_details[url] = {}
        if (i + 1) % 50 == 0:
            print(f"Enriched {i + 1}/{len(urls)} detail pages...")
        time.sleep(delay_s)

    # Merge into df
    detail_cols = [
        "lot_size", "street_size", "closest_landmark",
        "land_title", "detail_property_type", "halls", "balconies", "bathrooms", "rooms",
    ]
    for col in detail_cols:
        df[col] = df[url_col].map(lambda u: url_to_details.get(u, {}).get(col))

    print(f"Enriched {len(url_to_details)} listings with Overview & Details.")
    return df

In [9]:
# Enrich Voursa listings with Overview & Details (run this AFTER the Combine+export cell)
# Set ENRICH_MAX_ROWS = 20 for quick testing; None = all listings (~1s delay per page)

ENRICH_MAX_ROWS = None  # e.g. 20 for testing

# Check if df_all exists, otherwise try to load from CSV
if "df_all" not in globals() or df_all.empty:
    try:
        import pandas as pd
        df_all = pd.read_csv("real_estate_listings.csv")
        print(f"Loaded {len(df_all)} listings from CSV")
    except:
        print("No df_all found and CSV not available. Run the Combine+export cell first.")
        df_all = pd.DataFrame()

if not df_all.empty:
    # Check if already enriched
    detail_cols_check = ["lot_size", "street_size", "closest_landmark", "land_title", 
                        "detail_property_type", "halls", "balconies", "bathrooms", "rooms"]
    already_enriched = any(col in df_all.columns for col in detail_cols_check)
    
    if not already_enriched:
        print("Enriching listings with Overview & Details from detail pages...")
        df_all = enrich_df_with_details(df_all, session, delay_s=1.0, max_rows=ENRICH_MAX_ROWS)
    else:
        print("Listings already enriched. Skipping enrichment.")
    
    # Save to CSV
    out_csv = "real_estate_listings.csv"
    out_json = "real_estate_listings.jsonl"
    df_all.to_csv(out_csv, index=False)
    df_all.to_json(out_json, orient="records", lines=True, force_ascii=False)
    print(f"Saved: {out_csv} and {out_json}")
    
    # Show detail columns
    detail_cols = [c for c in detail_cols_check if c in df_all.columns]
    if detail_cols:
        print(f"\nDetail columns: {', '.join(detail_cols)}")
        display(df_all[["url"] + detail_cols].head(5))

No df_all found and CSV not available. Run the Combine+export cell first.


## Optional: scrape all State/City combinations via dropdown filters

Some real-estate sites expose a **Location** filter with two dropdowns (e.g. **State** and **City**).
The helper below is a **template** that uses Selenium to loop over every state/city combination,
load the results, and reuse our HTML parser to build one combined DataFrame.

> You will need to adjust the CSS/ID/XPath selectors and (optionally) the parser function
> to match the exact site you are targeting.

In [10]:
# Template: use Selenium to scrape all State/City combinations on a page

from typing import Callable

def scrape_all_state_city_with_selenium(
    url: str,
    parse_fn: Callable[[str], pd.DataFrame] = parse_voursa_category_html,
    state_locator: Tuple[str, str] = ("id", "state"),
    city_locator: Tuple[str, str] = ("id", "city"),
    wait_after_change: float = 1.5,
    max_states: Optional[int] = None,
    max_cities_per_state: Optional[int] = None,
) -> pd.DataFrame:
    """Generic helper to iterate over all state/city dropdown options.

    Parameters
    ----------
    url : str
        Page URL that contains the State/City dropdowns and the listings.
    parse_fn : callable
        Function that takes full HTML and returns a listings DataFrame.
        By default we reuse `parse_voursa_category_html`, but you can pass
        another parser.
    state_locator : (by, value)
        How to locate the *State* <select>. For example ("id", "state") or
        ("name", "StateId") or ("css", "select#state").
    city_locator : (by, value)
        How to locate the *City* <select>.
    wait_after_change : float
        Seconds to wait after changing a dropdown so that the page can
        refresh the results.
    max_states : Optional[int]
        If set, limit the number of state options (for testing).
    max_cities_per_state : Optional[int]
        If set, limit the number of city options per state (for testing).
    """

    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait, Select
    from selenium.webdriver.support import expected_conditions as EC

    by_map = {
        "id": By.ID,
        "name": By.NAME,
        "css": By.CSS_SELECTOR,
        "xpath": By.XPATH,
    }

    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1365,768")

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)

    def _find_select(locator: Tuple[str, str]) -> Select:
        by_key, value = locator
        by = by_map[by_key]
        el = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((by, value))
        )
        return Select(el)

    all_frames: List[pd.DataFrame] = []

    state_select = _find_select(state_locator)
    state_options = state_select.options

    print(f"Found {len(state_options)} state options (including placeholders)")

    state_count = 0
    for s_opt in state_options:
        value = s_opt.get_attribute("value")
        label = s_opt.text.strip()

        # Skip placeholder / empty options
        if not value:
            continue

        state_count += 1
        if max_states is not None and state_count > max_states:
            break

        print(f"Selecting state: {label} ({value})")
        state_select.select_by_value(value)
        time.sleep(wait_after_change)

        # Re-locate city select after state change
        city_select = _find_select(city_locator)
        city_options = city_select.options
        print(f"  -> {len(city_options)} city options for this state (incl. placeholder)")

        city_count = 0
        for c_opt in city_options:
            c_value = c_opt.get_attribute("value")
            c_label = c_opt.text.strip()

            if not c_value:
                continue

            city_count += 1
            if max_cities_per_state is not None and city_count > max_cities_per_state:
                break

            print(f"    Selecting city: {c_label} ({c_value})")
            city_select.select_by_value(c_value)
            time.sleep(wait_after_change)

            html = driver.page_source
            df = parse_fn(html)
            if df is None or df.empty:
                continue

            df["state_filter"] = label
            df["city_filter"] = c_label
            all_frames.append(df)

    driver.quit()

    if not all_frames:
        print("No listings collected from state/city iteration.")
        return pd.DataFrame()

    combined = pd.concat(all_frames, ignore_index=True)
    combined = combined.drop_duplicates(subset=["url"]) if "url" in combined.columns else combined
    print("Total listings from all state/city combinations:", len(combined))
    return combined


## Wassit scraper

Wassit may respond slowly or block aggressive scraping. The code below:

- Uses a session with retries
- Sends a realistic User-Agent
- Adds a small sleep between requests

If requests still fail, use Selenium (similar to Voursa) as a fallback.


In [11]:
# Wassit parsing (best-effort; you may need to tweak selectors)

WASSIT_URL = "https://www.wassit.info/immobilier.html"

def parse_wassit_html(html: str, base_url: str = "https://www.wassit.info") -> pd.DataFrame:
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    # The original notebook used div.center; keep it, but also try to grab per-ad links.
    # You'll likely want to inspect the HTML and refine this logic.
    for block in soup.select("div.center"):
        text = block.get_text(" ", strip=True)
        if not text:
            continue

        # Attempt to find the first link inside the block
        a = block.find("a", href=True)
        url = None
        if a:
            href = a["href"]
            url = href if href.startswith("http") else base_url.rstrip("/") + "/" + href.lstrip("/")

        rows.append({
            "source": "wassit",
            "url": url,
            "raw_text": text,
        })

    df = pd.DataFrame(rows)
    if not df.empty and "url" in df.columns:
        df = df.drop_duplicates(subset=["url"])
    return df

# Fetch Wassit
wassit_df = pd.DataFrame()
try:
    time.sleep(1.0)
    wassit_html = fetch_html(session, WASSIT_URL, timeout=25)
    wassit_df = parse_wassit_html(wassit_html)
    print("Wassit blocks:", len(wassit_df))
except Exception as e:
    print("Wassit fetch/parse failed:", repr(e))
    print("Tip: try running Selenium on Wassit, or reduce request frequency.")


Wassit fetch/parse failed: SSLError(MaxRetryError('HTTPSConnectionPool(host=\'www.wassit.info\', port=443): Max retries exceeded with url: /immobilier.html (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for \'www.wassit.info\'. (_ssl.c:1016)")))'))
Tip: try running Selenium on Wassit, or reduce request frequency.


In [12]:
# Combine + export

df_all = pd.concat(
    [
        voursa_df_full if 'voursa_df_full' in globals() else voursa_df,
        wassit_df if isinstance(wassit_df, pd.DataFrame) else pd.DataFrame(),
    ],
    ignore_index=True,
)

# Light cleaning
if "price_mru" in df_all.columns:
    df_all["price_mru"] = pd.to_numeric(df_all["price_mru"], errors="coerce")

df_all = df_all.drop_duplicates(subset=["source", "url"], keep="first")

# Enrich with Overview & Details from detail pages (optional)
# Set ENRICH_WITH_DETAILS = True to fetch detail pages (~1s per listing)
# Set ENRICH_MAX_ROWS = None for all listings, or a number (e.g., 20) for testing
ENRICH_WITH_DETAILS = True  # Set to True to enable enrichment
ENRICH_MAX_ROWS = None  # e.g., 20 for testing, None for all

if ENRICH_WITH_DETAILS and "url" in df_all.columns:
    print("Enriching listings with Overview & Details from detail pages...")
    df_all = enrich_df_with_details(df_all, session, delay_s=1.0, max_rows=ENRICH_MAX_ROWS)
    print("Enrichment complete.")

display(df_all.head(20))
print("Total listings:", len(df_all))

# Export
out_csv = "real_estate_listings.csv"
out_json = "real_estate_listings.jsonl"
df_all.to_csv(out_csv, index=False)
df_all.to_json(out_json, orient="records", lines=True, force_ascii=False)

print("Saved:", out_csv, "and", out_json)

# Show detail columns if they exist
detail_cols = [c for c in ["lot_size", "street_size", "closest_landmark", "land_title", "detail_property_type", "halls", "balconies", "bathrooms", "rooms"] if c in df_all.columns]
if detail_cols:
    print(f"\nDetail columns in CSV: {', '.join(detail_cols)}")
    display(df_all[["url"] + detail_cols].head(5))


Enriching listings with Overview & Details from detail pages...
Enriched 50/4050 detail pages...
Enriched 100/4050 detail pages...
Enriched 150/4050 detail pages...
Enriched 200/4050 detail pages...
Enriched 250/4050 detail pages...
Enriched 300/4050 detail pages...
Enriched 350/4050 detail pages...
Enriched 400/4050 detail pages...
Enriched 450/4050 detail pages...
Enriched 500/4050 detail pages...
Enriched 550/4050 detail pages...
Enriched 600/4050 detail pages...
Enriched 650/4050 detail pages...
Enriched 700/4050 detail pages...
Enriched 750/4050 detail pages...
Enriched 800/4050 detail pages...
Enriched 850/4050 detail pages...
Enriched 900/4050 detail pages...
Enriched 950/4050 detail pages...
Enriched 1000/4050 detail pages...
Enriched 1050/4050 detail pages...
Enriched 1100/4050 detail pages...
Enriched 1150/4050 detail pages...
Enriched 1200/4050 detail pages...
Enriched 1250/4050 detail pages...
Enriched 1300/4050 detail pages...
Enriched 1350/4050 detail pages...
Enriched 14

,raw_text,price_mru,property_type,time_ago,title_guess,location,source,url,lot_size,street_size,closest_landmark,land_title,detail_property_type,halls,balconies,bathrooms,rooms
0,Grand plaza Tevragh Zeina • 9 months ago Share...,2500,None,9 months ago,Grand plaza Tevragh Zeina • 9 months ago Share...,None,voursa,https://voursa.com/EN/categories/real_estate/a...,100.00,80.00,Palais des congrès,No,None,1.0,1.0,2.0,32.0
1,"نيمرو للبيــع فالمطار سوق السيارات 2,000,000 M...",2000000,Land Plot,4 hours ago,نيمرو للبيــع فالمطار سوق السيارات,Dar Naim,voursa,https://voursa.com/EN/categories/real_estate/a...,400.00,None,كرفور لابست,Yes,None,NaN,NaN,NaN,NaN
2,"سكن فبوحديده احذ كدروه حظ سعيد 550,000 MRU Res...",550000,Residential Property,4 hours ago,سكن فبوحديده احذ كدروه حظ سعيد,Toujounine,voursa,https://voursa.com/EN/categories/real_estate/a...,150.00,10.00,كدروه حظ سعيد,No,Other,1.0,1.0,1.0,3.0
3,"فلبوادي على شارع كبير 300م 1,400,000 MRU Land ...",1400000,Land Plot,4 hours ago,فلبوادي على شارع كبير 300م,Tevragh Zeina,voursa,https://voursa.com/EN/categories/real_estate/a...,300.00,20.00,كدروه البوادي,Yes,None,NaN,NaN,NaN,NaN
4,"ديبلكس اكبير للبيع 15,000,000 MRU Residential ...",15000000,Residential Property,6 hours ago,ديبلكس اكبير للبيع,Tevragh Zeina,voursa,https://voursa.com/EN/categories/real_estate/a...,930.00,None,سكتير 5,Yes,Duplex,0.0,0.0,0.0,0.0
5,"دار فعرفات الداية 11 1,300,000 MRU Residential...",1300000,Residential Property,8 hours ago,دار فعرفات الداية 11,Arafat,voursa,https://voursa.com/EN/categories/real_estate/a...,180.00,20.00,بقالة الحوضين,Yes,Apartment,1.0,1.0,2.0,3.0
6,"ديبلكس للبيع 3,300,000 MRU Residential Propert...",3300000,Residential Property,8 hours ago,ديبلكس للبيع,Teyarett,voursa,https://voursa.com/EN/categories/real_estate/a...,180.00,8.00,سوكوجيم عين الطلح,Yes,Duplex,0.0,0.0,0.0,0.0
7,"نيمرو للبيــع في سكتير 5 F Nord 6,700,000 MRU ...",6700000,Land Plot,9 hours ago,نيمرو للبيــع في سكتير 5 F Nord,Tevragh Zeina,voursa,https://voursa.com/EN/categories/real_estate/a...,875.00,None,سكتير 5,Yes,None,NaN,NaN,NaN,NaN
8,"150م² فكرن فشرم الشيخ البوادي نظيفة 1,800,000 ...",1800000,Land Plot,9 hours ago,150م² فكرن فشرم الشيخ البوادي نظيفة,Teyarett,voursa,https://voursa.com/EN/categories/real_estate/a...,150.00,None,البوادي,No,None,NaN,NaN,NaN,NaN
9,"ديبلكس للبيع 8,500,000 MRU Residential Propert...",8500000,Residential Property,9 hours ago,ديبلكس للبيع,Teyarett,voursa,https://voursa.com/EN/categories/real_estate/a...,400.00,None,الصحراوي,Yes,Duplex,0.0,0.0,0.0,0.0


Total listings: 4050
Saved: real_estate_listings.csv and real_estate_listings.jsonl

Detail columns in CSV: lot_size, street_size, closest_landmark, land_title, detail_property_type, halls, balconies, bathrooms, rooms


,url,lot_size,street_size,closest_landmark,land_title,detail_property_type,halls,balconies,bathrooms,rooms
0,https://voursa.com/EN/categories/real_estate/a...,100.00,80.00,Palais des congrès,No,None,1.0,1.0,2.0,32.0
1,https://voursa.com/EN/categories/real_estate/a...,400.00,None,كرفور لابست,Yes,None,NaN,NaN,NaN,NaN
2,https://voursa.com/EN/categories/real_estate/a...,150.00,10.00,كدروه حظ سعيد,No,Other,1.0,1.0,1.0,3.0
3,https://voursa.com/EN/categories/real_estate/a...,300.00,20.00,كدروه البوادي,Yes,None,NaN,NaN,NaN,NaN
4,https://voursa.com/EN/categories/real_estate/a...,930.00,None,سكتير 5,Yes,Duplex,0.0,0.0,0.0,0.0


In [13]:
# Quick sanity checks / mini-EDA

if not df_all.empty:
    if "location" in df_all.columns:
        print(df_all["location"].value_counts(dropna=False).head(15))
    if "property_type" in df_all.columns:
        print(df_all["property_type"].value_counts(dropna=False).head(15))
    if "price_mru" in df_all.columns:
        print(df_all["price_mru"].describe())


location
Tevragh Zeina    2205
Teyarett          695
Dar Naim          450
Arafat            331
Toujounine        186
Ksar               72
Riyadh             36
None               34
Nouadhibou         23
Sebkha             16
Chami               2
Name: count, dtype: int64
property_type
Land Plot                                2479
Residential Property                     1332
Commercial and Industrial Real Estate     133
Store                                      50
Other                                      28
Office                                     23
Warehouse                                   4
None                                        1
Name: count, dtype: int64
count    4.050000e+03
mean     4.437509e+06
std      2.614544e+07
min      1.000000e+00
25%      1.150000e+06
50%      2.700000e+06
75%      4.600000e+06
max      1.500000e+09
Name: price_mru, dtype: float64


In [16]:
# Transform scraper data to raw.csv format

def transform_to_raw_format(df: pd.DataFrame) -> pd.DataFrame:
    """Transform scraper DataFrame to match raw.csv column structure.
    
    Mapping:
    - titre -> title_guess
    - type_bien -> detail_property_type (fallback to property_type)
    - type_annonce -> "Vente" (default, could be enhanced)
    - prix -> price_mru
    - surface_m2 -> lot_size
    - nb_chambres -> rooms
    - nb_salons -> halls
    - nb_sdb -> bathrooms
    - quartier -> location
    - ville -> derived from location (or None)
    - description -> raw_text
    - source -> source
    - date_publication -> time_ago
    - caracteristiques -> combined features
    - url -> url
    """
    
    # Mapping state to city (for ville column)
    STATE_TO_CITY = {
        "Tevragh Zeina": "Nouakchott Ouest",
        "Dar Naim": "Nouakchott Nord",
        "Arafat": "Nouakchott Sud",
        "Teyarett": "Nouakchott Nord",
        "Ksar": "Nouakchott Ouest",
        "Riyadh": "Nouakchott Sud",
        "EL Mina": "Nouakchott Sud",
        "Sebkha": "Nouakchott Ouest",
        "Toujounine": "Nouakchott Nord",
        "Nouadhibou": "Dakhlet Nouadhibou",
        "Chami": "Dakhlet Nouadhibou",
    }
    
    result = pd.DataFrame()
    
    # Direct mappings with safe column access
    result["titre"] = df["title_guess"] if "title_guess" in df.columns else ""
    
    # type_bien: use detail_property_type, fallback to property_type
    if "detail_property_type" in df.columns and "property_type" in df.columns:
        result["type_bien"] = df["detail_property_type"].fillna(df["property_type"])
    elif "detail_property_type" in df.columns:
        result["type_bien"] = df["detail_property_type"]
    elif "property_type" in df.columns:
        result["type_bien"] = df["property_type"]
    else:
        result["type_bien"] = ""
    
    result["type_annonce"] = "Vente"  # Default, could be enhanced based on data
    result["prix"] = df["price_mru"] if "price_mru" in df.columns else None
    result["surface_m2"] = df["lot_size"] if "lot_size" in df.columns else None
    result["nb_chambres"] = df["rooms"] if "rooms" in df.columns else None
    result["nb_salons"] = df["halls"] if "halls" in df.columns else None
    result["nb_sdb"] = df["bathrooms"] if "bathrooms" in df.columns else None
    result["quartier"] = df["location"] if "location" in df.columns else ""
    result["description"] = df["raw_text"] if "raw_text" in df.columns else ""
    result["source"] = df["source"] if "source" in df.columns else ""
    result["date_publication"] = df["time_ago"] if "time_ago" in df.columns else ""
    result["url"] = df["url"] if "url" in df.columns else ""
    
    # Derive ville from location
    if "location" in df.columns:
        result["ville"] = df["location"].map(STATE_TO_CITY)
    else:
        result["ville"] = ""
    
    # Combine caracteristiques for each row
    def combine_features(row):
        features = []
        if "closest_landmark" in row.index and pd.notna(row["closest_landmark"]):
            features.append(f"landmark: {row['closest_landmark']}")
        if "land_title" in row.index and pd.notna(row["land_title"]):
            features.append(f"land_title: {row['land_title']}")
        if "balconies" in row.index and pd.notna(row["balconies"]):
            features.append(f"balconies: {int(row['balconies'])}")
        if "street_size" in row.index and pd.notna(row["street_size"]):
            features.append(f"street_size: {row['street_size']}")
        return ", ".join(features) if features else ""
    
    result["caracteristiques"] = df.apply(combine_features, axis=1)
    
    # Ensure correct column order
    column_order = [
        "titre", "type_bien", "type_annonce", "prix", "surface_m2",
        "nb_chambres", "nb_salons", "nb_sdb", "quartier", "ville",
        "description", "source", "date_publication", "caracteristiques", "url"
    ]
    
    return result[column_order]

# Example: transform current df_all to raw format
if "df_all" in globals() and not df_all.empty:
    df_raw = transform_to_raw_format(df_all)
    print(f"Transformed {len(df_raw)} listings to raw.csv format")
    print("\nColumn mapping:")
    print("  titre -> title_guess")
    print("  type_bien -> detail_property_type / property_type")
    print("  type_annonce -> 'Vente' (default)")
    print("  prix -> price_mru")
    print("  surface_m2 -> lot_size")
    print("  nb_chambres -> rooms")
    print("  nb_salons -> halls")
    print("  nb_sdb -> bathrooms")
    print("  quartier -> location")
    print("  ville -> derived from location")
    print("  description -> raw_text")
    print("  source -> source")
    print("  date_publication -> time_ago")
    print("  caracteristiques -> combined features")
    print("  url -> url")
    
    display(df_raw.head(10))
    
    # Save to raw.csv
    df_raw.to_csv("raws.csv", index=False)
    print("\n✅ Saved to raw.csv")
else:
    # Try loading from CSV if df_all not available
    try:
        df_all = pd.read_csv("real_estate_listings.csv")
        df_raw = transform_to_raw_format(df_all)
        df_raw.to_csv("raw.csv", index=False)
        print(f"✅ Transformed {len(df_raw)} listings from CSV and saved to raw.csv")
        display(df_raw.head(10))
    except Exception as e:
        print(f"Error: {e}")
        print("Please run the Combine+export cell first, or ensure real_estate_listings.csv exists.")

Transformed 4050 listings to raw.csv format

Column mapping:
  titre -> title_guess
  type_bien -> detail_property_type / property_type
  type_annonce -> 'Vente' (default)
  prix -> price_mru
  surface_m2 -> lot_size
  nb_chambres -> rooms
  nb_salons -> halls
  nb_sdb -> bathrooms
  quartier -> location
  ville -> derived from location
  description -> raw_text
  source -> source
  date_publication -> time_ago
  caracteristiques -> combined features
  url -> url


,titre,type_bien,type_annonce,prix,surface_m2,nb_chambres,nb_salons,nb_sdb,quartier,ville,description,source,date_publication,caracteristiques,url
0,Grand plaza Tevragh Zeina • 9 months ago Share...,None,Vente,2500,100.00,32.0,1.0,2.0,None,NaN,Grand plaza Tevragh Zeina • 9 months ago Share...,voursa,9 months ago,"landmark: Palais des congrès, land_title: No, ...",https://voursa.com/EN/categories/real_estate/a...
1,نيمرو للبيــع فالمطار سوق السيارات,Land Plot,Vente,2000000,400.00,NaN,NaN,NaN,Dar Naim,Nouakchott Nord,"نيمرو للبيــع فالمطار سوق السيارات 2,000,000 M...",voursa,4 hours ago,"landmark: كرفور لابست, land_title: Yes",https://voursa.com/EN/categories/real_estate/a...
2,سكن فبوحديده احذ كدروه حظ سعيد,Other,Vente,550000,150.00,3.0,1.0,1.0,Toujounine,Nouakchott Nord,"سكن فبوحديده احذ كدروه حظ سعيد 550,000 MRU Res...",voursa,4 hours ago,"landmark: كدروه حظ سعيد, land_title: No, balco...",https://voursa.com/EN/categories/real_estate/a...
3,فلبوادي على شارع كبير 300م,Land Plot,Vente,1400000,300.00,NaN,NaN,NaN,Tevragh Zeina,Nouakchott Ouest,"فلبوادي على شارع كبير 300م 1,400,000 MRU Land ...",voursa,4 hours ago,"landmark: كدروه البوادي, land_title: Yes, stre...",https://voursa.com/EN/categories/real_estate/a...
4,ديبلكس اكبير للبيع,Duplex,Vente,15000000,930.00,0.0,0.0,0.0,Tevragh Zeina,Nouakchott Ouest,"ديبلكس اكبير للبيع 15,000,000 MRU Residential ...",voursa,6 hours ago,"landmark: سكتير 5, land_title: Yes, balconies: 0",https://voursa.com/EN/categories/real_estate/a...
5,دار فعرفات الداية 11,Apartment,Vente,1300000,180.00,3.0,1.0,2.0,Arafat,Nouakchott Sud,"دار فعرفات الداية 11 1,300,000 MRU Residential...",voursa,8 hours ago,"landmark: بقالة الحوضين, land_title: Yes, balc...",https://voursa.com/EN/categories/real_estate/a...
6,ديبلكس للبيع,Duplex,Vente,3300000,180.00,0.0,0.0,0.0,Teyarett,Nouakchott Nord,"ديبلكس للبيع 3,300,000 MRU Residential Propert...",voursa,8 hours ago,"landmark: سوكوجيم عين الطلح, land_title: Yes, ...",https://voursa.com/EN/categories/real_estate/a...
7,نيمرو للبيــع في سكتير 5 F Nord,Land Plot,Vente,6700000,875.00,NaN,NaN,NaN,Tevragh Zeina,Nouakchott Ouest,"نيمرو للبيــع في سكتير 5 F Nord 6,700,000 MRU ...",voursa,9 hours ago,"landmark: سكتير 5, land_title: Yes",https://voursa.com/EN/categories/real_estate/a...
8,150م² فكرن فشرم الشيخ البوادي نظيفة,Land Plot,Vente,1800000,150.00,NaN,NaN,NaN,Teyarett,Nouakchott Nord,"150م² فكرن فشرم الشيخ البوادي نظيفة 1,800,000 ...",voursa,9 hours ago,"landmark: البوادي, land_title: No",https://voursa.com/EN/categories/real_estate/a...
9,ديبلكس للبيع,Duplex,Vente,8500000,400.00,0.0,0.0,0.0,Teyarett,Nouakchott Nord,"ديبلكس للبيع 8,500,000 MRU Residential Propert...",voursa,9 hours ago,"landmark: الصحراوي, land_title: Yes, balconies: 0",https://voursa.com/EN/categories/real_estate/a...



✅ Saved to raw.csv


In [ ]:
# Diagnose why columns are null in raws.csv

import pandas as pd

# Load source data
df_source = pd.read_csv("real_estate_listings.csv")
print(f"Total listings: {len(df_source)}")
print("\nDetail columns non-null counts:")
detail_cols = ["lot_size", "rooms", "halls", "bathrooms", "street_size", "closest_landmark", "land_title", "balconies"]
for col in detail_cols:
    if col in df_source.columns:
        non_null = df_source[col].notna().sum()
        pct = (non_null / len(df_source)) * 100
        print(f"  {col:20s}: {non_null:5d} ({pct:5.1f}%)")
    else:
        print(f"  {col:20s}: MISSING COLUMN")

print("\n⚠️  REASON FOR NULL VALUES:")
print("The detail columns (rooms, halls, bathrooms, etc.) are null because:")
print("1. Only 20 listings were enriched (ENRICH_MAX_ROWS = 20 was set for testing)")
print("2. Many detail pages don't have these fields in their JSON data")
print("3. To get more data, run the enrichment cell with ENRICH_MAX_ROWS = None")
print("\n💡 SOLUTION:")
print("To populate these columns, you need to:")
print("1. Go to the enrichment cell (cell 10)")
print("2. Set ENRICH_MAX_ROWS = None (to enrich all listings)")
print("3. Run the cell (this will take ~1 hour for 4000 listings)")
print("4. Then re-run the transformation to raw.csv format")

In [17]:
# Quick script to add detail columns to CSV
# This cell loads the CSV, enriches it, and saves it back

import pandas as pd

# Load CSV
print("Loading CSV...")
df_all = pd.read_csv("real_estate_listings.csv")
print(f"Loaded {len(df_all)} listings")

# Check if already enriched
detail_cols_check = ["lot_size", "street_size", "closest_landmark", "land_title", 
                    "detail_property_type", "halls", "balconies", "bathrooms", "rooms"]
already_enriched = any(col in df_all.columns for col in detail_cols_check)

if already_enriched:
    print("CSV already has detail columns!")
    print(f"Existing detail columns: {[c for c in detail_cols_check if c in df_all.columns]}")
else:
    print("Enriching CSV with Overview & Details columns...")
    print("This will take approximately 1 hour for all listings.")
    print("Set ENRICH_MAX_ROWS below to limit for testing (e.g., 20)")
    
    ENRICH_MAX_ROWS = 20  # Set to 20 for quick test, None for all
    
    # Ensure session exists
    if 'session' not in globals():
        session = make_session()
    
    # Enrich
    df_all = enrich_df_with_details(df_all, session, delay_s=1.0, max_rows=ENRICH_MAX_ROWS)
    
    # Save
    df_all.to_csv("real_estate_listings.csv", index=False)
    df_all.to_json("real_estate_listings.jsonl", orient="records", lines=True, force_ascii=False)
    print("\n CSV updated with detail columns!")
    
    # Show summary
    detail_cols = [c for c in detail_cols_check if c in df_all.columns]
    print(f"\nDetail columns added: {', '.join(detail_cols)}")
    display(df_all[["url"] + detail_cols].head(5))

Loading CSV...
Loaded 4050 listings
CSV already has detail columns!
Existing detail columns: ['lot_size', 'street_size', 'closest_landmark', 'land_title', 'detail_property_type', 'halls', 'balconies', 'bathrooms', 'rooms']
